# Hybrid Architecture Decision Report

A hybrid approach combines retrieved evidence with learned or prompted behavior. Here we run RAG-only, style-only, and hybrid prompts on the same requests, score the answers locally with one rubric, and turn the results into an architecture recommendation.

## Learning Objectives

By the end of this notebook, you should be able to:

- Define freshness, auditability, consistency, latency, cost, and maintenance criteria.
- Build a hybrid prompt from retrieved policies and style examples.
- Compare RAG-only, style-only, and hybrid answers with one rubric.
- Check a model-written decision report against the measured results.


## Decision Criteria

A useful architecture decision is not just a winner label. It says what evidence was tested, which risks remain, and what trigger would justify revisiting the decision. For RAG and fine-tuning choices, these criteria matter most:

| Criterion | Meaning | What affects it |
|---|---|---|
| **Freshness** | The system reflects updated policy without retraining | RAG wins: documents can be updated and re-indexed |
| **Auditability** | A reviewer can trace an answer back to evidence | Citations and source IDs |
| **Behavior consistency** | The same tone, format, and refusal style across similar requests | Fine-tuning or strong examples |
| **Latency** | The time needed to answer | Retrieval and longer prompts add overhead |
| **Cost** | Provider calls, embeddings, vector storage, and training | Every added component and call |
| **Maintenance effort** | Keeping documents current, reviewing examples, monitoring failures | The number of moving parts |

A hybrid strategy should not be the default just because it sounds complete. Start with the simplest approach that satisfies the criteria. Add style conditioning or fine-tuning only when the measured failures point to behavior adaptation rather than missing evidence.

## Does Combining Strategies Always Help?

A hybrid sounds like the safest choice, but measurements do not always support it.

The chart below comes from a study that added new knowledge to three 7B models with fine-tuning (FT), RAG, or both. RAG gave a large gain on every model, fine-tuning alone helped little, and combining them beat RAG alone only for Llama2. That study combined RAG with fine-tuning, while this notebook combines RAG with style examples, but the lesson carries over: treat a hybrid as a hypothesis to measure, not a default.

<div align="center">
  <img src="assets/05-knowledge-injection-gains.png" alt="Mean accuracy gain of fine-tuning, RAG, and both across three models" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Mean accuracy gain from fine-tuning (FT), RAG, and both (FT+RAG) on knowledge-injection tasks. Source: <a href="https://arxiv.org/abs/2312.05934" target="_blank" rel="noopener noreferrer">Ovadia et al., 2023</a>, CC BY 4.0</em></p>
</div>


## Load Shared Data And Runtime Helpers


In [ ]:
import json
import re
from pprint import pprint
from typing import Any, Literal

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ModelReply,
    ReplySource,
    call_model,
    format_context,
    retrieve,
    style_messages,
)

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | Model: {CONFIG.model} | Dry run: {CONFIG.dry_run}"
)

## Reuse The Evaluated Retrieval Baseline

The hybrid comparison reuses the same ranked-search helper as the RAG-only notebook. Keeping retrieval fixed means later differences come from behavior conditioning rather than a different evidence path.


In [ ]:
retrieval_preview = retrieve(REQUESTS[0]["question"], top_k=3)
[(row["doc_id"], round(row["score"], 3)) for row in retrieval_preview]

## Reuse Approved Style Examples

The shared helper converts approved examples into alternating user and assistant messages. These are the same examples as in the fine-tuning-style baseline, so the behavior signal comes from the same data.


In [ ]:
few_shot_preview = style_messages(STYLE_EXAMPLES[:2])
for message in few_shot_preview:
    print(f"{message['role']}: {message['content']}")

The hybrid prompt asks for a policy ID after every sentence, but these replies contain none, and the model tends to copy the examples' format along with their tone. So for the hybrid condition, we add each example's policy ID to its reply: the examples should demonstrate the exact behavior we want repeated, citations included. The style-only condition keeps the uncited replies, because it receives no policy context to cite.

In [ ]:
EXAMPLE_SOURCES = {
    "refund": "refund-policy",
    "technical": "api-rate-limit",
    "security": "mfa-reset",
    "programme": "extension-policy",
}


def cite_example(example: dict[str, str]) -> dict[str, str]:
    """Add the example's policy ID before the end of each sentence of its reply."""
    doc_id = EXAMPLE_SOURCES[example["intent"]]
    sentences = re.split(r"(?<=[.?!])\s+", example["assistant"])
    cited_sentences = []
    for sentence in sentences:
        punctuation = sentence[-1:] if sentence.endswith((".", "?", "!")) else ""
        text = sentence[:-1] if punctuation else sentence
        cited_sentences.append(f"{text} [{doc_id}]{punctuation}")
    cited = " ".join(cited_sentences)
    return {**example, "assistant": cited}


cited_style_examples = [cite_example(example) for example in STYLE_EXAMPLES]
print(cited_style_examples[0]["assistant"])

## Build The Hybrid Prompt

The hybrid prompt includes the top retrieved policy records plus examples of the desired response behavior. This is the pattern to compare against RAG-only and style-only prompts.


All three conditions share one answer schema. The application, not the model, records whether a reply was live, simulated, or a fallback, and the scores are computed locally below instead of asking the model to rate itself.

In [ ]:
class StrategyResult(BaseModel):
    """One answer plus application-owned execution and parsing metadata."""

    answer: str
    source_ids: list[str]
    response_source: ReplySource = "simulated"
    parse_valid: bool = True

Keep one prompt builder and vary only the retrieved context and the style examples (uncited for style-only, cited for hybrid). Every condition returns JSON with `answer` and `source_ids`.

In [ ]:
def strategy_messages(
    request: dict[str, Any], evidence: list[dict[str, Any]], strategy: str
) -> list[dict[str, str]]:
    """Keep output instructions identical across the three prompting conditions."""
    examples = {
        "rag": [],
        "style_only": style_messages(STYLE_EXAMPLES),
        "hybrid": style_messages(cited_style_examples),
    }[strategy]
    return [
        {
            "role": "system",
            "content": (
                "Answer only from supplied policy context. If it is insufficient, say so. "
                "Return JSON with answer and source_ids. "
                "End each sentence of the answer with the ID of the policy it comes from in square brackets, "
                'for example: "Late fees apply after 14 days [billing-terms]." '
                "List the same IDs in source_ids. Do not invent IDs."
            ),
        },
        *examples,
        {
            "role": "user",
            "content": f"Context:\n{format_context(evidence) or 'No policy context supplied.'}\n\nQuestion: {request['question']}",
        },
    ]

Without a live response, use a clearly labeled illustrative answer that copies the top retrieved document. It never looks at the expected document, so it cannot manufacture a correct score.

In [ ]:
def simulated_response(evidence: list[dict[str, Any]]) -> str:
    """Supply a transparent example, not evidence of model performance."""
    if not evidence:
        return json.dumps(
            {"answer": "I need current policy evidence to answer.", "source_ids": []}
        )
    first = evidence[0]
    return json.dumps(
        {
            "answer": f"{first['text']} [{first['doc_id']}]",
            "source_ids": [first["doc_id"]],
        }
    )

Parse the reply and preserve whether it was live, simulated, or a fallback. Malformed JSON is a failed format check, not an answer to score as successful.

In [ ]:
def strategy_answer(request: dict[str, Any], strategy: str) -> StrategyResult:
    """Run one condition against the same query and output contract."""
    evidence = (
        [] if strategy == "style_only" else retrieve(request["question"], top_k=3)
    )
    reply = call_model(
        strategy_messages(request, evidence, strategy),
        dry_response=simulated_response(evidence),
        response_format={"type": "json_object"},
    )
    try:
        result = StrategyResult.model_validate_json(reply.text)
        result.response_source = reply.source
        result.parse_valid = True
        return result
    except ValidationError:
        return StrategyResult(
            answer="", source_ids=[], response_source=reply.source, parse_valid=False
        )

Run one hybrid call before building the strategy table so the output shape is visible.


In [ ]:
result = strategy_answer(REQUESTS[2], "hybrid")
pprint(result.model_dump())

## Compare All Three Conditions With One Rubric

For each held-out request, run RAG-only, style-only, and hybrid prompting. The common checks measure parsing, retrieval coverage, citation-ID consistency, expected-source citation, and response length. None is a semantic faithfulness or tone judge.

In [ ]:
def score_answer(
    request: dict[str, Any], strategy: str, result: StrategyResult
) -> dict[str, Any]:
    """Apply identical observable checks without asking the model to grade itself."""
    evidence = (
        [] if strategy == "style_only" else retrieve(request["question"], top_k=3)
    )
    retrieved_ids = {item["doc_id"] for item in evidence}
    inline_ids = set(re.findall(r"\[([^\[\]]*)\]", result.answer))
    declared_ids = set(result.source_ids)
    citation_ids_valid = (
        result.parse_valid
        and bool(inline_ids)
        and inline_ids == declared_ids
        and inline_ids.issubset(retrieved_ids)
    )
    return {
        "request_id": request["request_id"],
        "strategy": strategy,
        "response_source": result.response_source,
        "answer": result.answer,
        "format_valid": result.parse_valid,
        "retrieval_hit": request["expected_doc"] in retrieved_ids,
        "citation_ids_valid": citation_ids_valid,
        "expected_source_cited": citation_ids_valid
        and request["expected_doc"] in declared_ids,
        "concise": bool(result.answer.strip()) and len(result.answer.split()) <= 80,
    }


strategy_rows = [
    score_answer(request, strategy, strategy_answer(request, strategy))
    for request in REQUESTS
    for strategy in ("rag", "style_only", "hybrid")
]
print(
    f"{'request':<9}{'strategy':<12}{'source':<11}"
    f"{'format':<8}{'retrieval':<11}{'cite IDs':<11}{'expected':<10}concise"
)
for row in strategy_rows:
    print(
        f"{row['request_id']:<9}{row['strategy']:<12}{row['response_source']:<11}"
        f"{row['format_valid']!s:<8}{row['retrieval_hit']!s:<11}"
        f"{row['citation_ids_valid']!s:<11}{row['expected_source_cited']!s:<10}"
        f"{row['concise']}"
    )

print()
for row in strategy_rows:
    print(f"{row['request_id']} {row['strategy']}: {row['answer']}")

## Interpreting The Comparison Table

`retrieval_hit` checks whether the expected policy entered the context. `citation_ids_valid` requires matching inline and declared IDs drawn from that context; it does not measure whether every sentence is cited. `expected_source_cited` checks whether the labeled policy was cited. `concise` only checks a maximum of 80 words, not tone or correctness.

Inspect the actual answers too. A correct source ID does not prove its claims. Rows marked `simulated` or `fallback` demonstrate the rubric, but are excluded from the live architecture recommendation.

All three conditions share the same system prompt, including "Answer only from supplied policy context". That is why style-only answers here usually say the context is insufficient, while in notebook 04 the style-only prompt let the model answer from general knowledge.

Live results also vary from run to run. With six requests, one or two answers that drop a citation can change which strategy looks best, so we run the comparison more than once before trusting a difference between strategies.

### Exercise: Summarize Live Rows Per Strategy

The decision report may only use live answers. For each strategy, count the live rows in `strategy_rows` and how many of them cite the expected source. Skip rows whose `response_source` is `simulated` or `fallback`.


In [ ]:
# @TODO: Count live rows and expected-source citations per strategy.


<details>
<summary>Show Reference Solution</summary>

```python
live_summary = {}
for row in strategy_rows:
    if row["response_source"] != "live":
        continue
    counts = live_summary.setdefault(
        row["strategy"], {"live_rows": 0, "expected_source_cited": 0}
    )
    counts["live_rows"] += 1
    counts["expected_source_cited"] += row["expected_source_cited"]
live_summary
```

</details>


Filtering changes the picture. Simulated and fallback rows reuse a deterministic answer that copies the top retrieved document with a valid citation, so they often pass every check. Counting them would make a strategy look better than the live model performed. If a strategy has few or no live rows, for example after rate-limit errors, the report should say that it was not measured rather than pick a winner. In a production evaluation, we would also count how often each strategy fell back: the answers are not model output, but the failures are a real reliability signal.


## Produce The Decision Report

A recommendation follows from the measured failures:

- If retrieval grounding is high and style is acceptable → recommend RAG-only because it is simpler and easier to maintain
- If retrieval grounding is high but tone is inconsistent → recommend a hybrid prompt with better examples before scheduling fine-tuning
- If retrieval grounding is low → improve chunking, query rewriting, or the corpus before changing model behavior

The rubric does not judge tone, so we decide whether style is acceptable by reading the answers, or with a tone check like the one in notebook 04.

Fine-tuning becomes a stronger recommendation only after repeated behavior failures are measured on held-out examples, and even then it should be compared against a stronger prompt baseline. Few-shot prompting does not establish whether trained LoRA weights would help.

The report should explain what was tried, why the chosen path is sufficient for now, and what signal would trigger the next experiment. Only a complete live comparison can support it: if any answer was simulated or fell back after an API error, the report says that nothing was measured instead of picking a winner.

The reviewer receives the rubric results and the actual answers, and returns a recommended strategy, a rationale, and a revisit trigger, which we validate before reading.

In [ ]:
class DecisionReport(BaseModel):
    """One recommendation for the whole comparison."""

    recommended_strategy: Literal["rag", "style_only", "hybrid", "not evaluated"]
    rationale: str
    revisit_trigger: str


live_comparison_complete = all(
    row["response_source"] == "live" for row in strategy_rows
)
decision_messages = [
    {
        "role": "system",
        "content": (
            "Review these common-rubric results and actual answers, and recommend one strategy "
            "for the whole comparison. Return one JSON object with exactly three keys: "
            "recommended_strategy (rag, style_only, or hybrid), rationale, and revisit_trigger. "
            "Citation-ID consistency proves neither sentence-level coverage nor faithfulness; concision does not prove tone. "
            "These are prompts, not trained fine-tuned models."
        ),
    },
    {"role": "user", "content": json.dumps(strategy_rows, indent=2)},
]

The fallback report distinguishes an unmeasured comparison from a live comparison whose reviewer is unavailable.

In [ ]:
dry_decision = json.dumps(
    {
        "recommended_strategy": "not evaluated",
        "rationale": (
            "The comparison was live, but no live reviewer report is available. The measured rows remain available for manual review."
            if live_comparison_complete
            else "A complete live comparison was not measured; simulated and fallback answers are illustrative only."
        ),
        "revisit_trigger": (
            "Inspect the measured answers and rubric manually, or retry the reviewer, before choosing a strategy."
            if live_comparison_complete
            else "Run the same rubric on live answers and inspect policy claims before choosing a strategy."
        ),
    },
    indent=2,
)

Call the reviewer only after a complete live comparison. Otherwise, show the dry report without an extra provider call.

In [ ]:
decision_reply = (
    call_model(
        decision_messages,
        dry_response=dry_decision,
        response_format={"type": "json_object"},
    )
    if live_comparison_complete
    else ModelReply(text=dry_decision, source="simulated")
)
print(f"Report ({decision_reply.source}):")

try:
    report = DecisionReport.model_validate_json(decision_reply.text)
    pprint(report.model_dump())
except ValidationError as error:
    print("The report does not match the requested shape:")
    print(error)

Check the report against the comparison table before accepting it. A model reviewer can recommend a strategy whose own rows failed the checks, for example a hybrid whose answers dropped their citations. The rubric results are the evidence; the report is a draft we verify.

## Check Your Understanding

1. Which risk does citation auditability address: freshness, grounding, or tone?
2. What two signals does the hybrid strategy combine?
3. What evidence would justify moving from prompt improvements to real fine-tuning?
4. Why do we check the reviewer's report against the comparison table before accepting it?

<details>
<summary>Show Reference Answers</summary>

1. Citation auditability mainly addresses grounding because reviewers can trace an answer back to retrieved evidence.
2. It combines retrieved evidence for grounding with style examples for response behavior.
3. Repeated style or formatting failures on held-out evaluation examples would justify a real fine-tuning experiment.
4. The report is model output too and can contradict the table, so the table is the evidence.

</details>


## Further Reading

- [RAFT: Adapting Language Model to Domain Specific RAG](https://arxiv.org/abs/2403.10131) - Zhang et al., 2024
